In [0]:
fact_billing = spark.sql(f"select * from regis_healthcare.silver.billing;")
fact_billing.createOrReplaceTempView("billing")

In [0]:
# Fact_Billing -- > Source: billing
# | Foreign Keys     |
# | ---------------- |
# | billing_key      |
# | resident_key     |
# | facility_key     |
# | billing_type_key |
# | billing_date_key |
# | payment_date_key |

# display(df_billing)
# display(Dim_Billing)
from pyspark.sql.functions import col, when,date_format
#-------------

# fact_billing.createOrReplaceTempView("dim")

# fact_billingS = spark.sql("""
# select t.*,d.billing_type_key from dim as d join  billing as t on d.billing_type = t.billing_type""")
#------------
# Create date_key column in YYYYMMDD format
fact_billing = fact_billing.withColumn("billing_type_key",when(col("billing_type")== "ACCOMMODATION",1)
.when(col("billing_type")== "ALLIED HEALTH",2)
.when(col("billing_type")== "CARE FEE",3)
.when(col("billing_type")== "MEDICATION",4)
.when(col("billing_type")== "NOT PROVIDE",5)
.when(col("billing_type")== "OTHER",6)
.when(col("billing_type")== "TRANSPORT",7))

fact_billing = fact_billing.withColumn("payment_date_key", date_format(col("payment_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_billing = fact_billing.withColumn("payment_date_key", col("payment_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
fact_billing = fact_billing.withColumn(
    "billing_key",
    regexp_replace(col("billing_id"), "^BIL", "").cast("int")
)
fact_billing = fact_billing.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)

fact_billing = fact_billing.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# Create date_key column in YYYYMMDD format
fact_billing = fact_billing.withColumn("billing_date_key", date_format(col("billing_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_billing = fact_billing.withColumn("billing_date_key", col("billing_date_key").cast("int"))
# #-------------------------
# display(fact_billing)
fact_billing = fact_billing.select(
 "billing_key" ,     
 "resident_key" ,    
 "facility_key"  ,   
 "billing_type_key", 
 "billing_date_key" ,
 "payment_date_key" 
)

display(fact_billing)



#### cataloge 

In [0]:
fact_billing.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_billing")

In [0]:
fact_billing.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_billing")
print(fact_billing.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_billing")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_billing")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.billing_key = source.billing_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_billing;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_billing;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_billing.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_billing")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_billing"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_billing

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.billing_key = source.billing_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
